# TabTransformer 기반 TAT 예측 모델

## 📌 개요
TabTransformer는 범주형과 연속형 변수를 효과적으로 결합하여 처리하는 Transformer 기반 예측 모델입니다. 공정 데이터의 Turn Around Time(TAT)을 예측하기 위해 범주형 특성(oper_group, days, shift 등)과 연속형 특성(x2-x21)을 동시에 활용합니다.

**데이터 구조**: 일반적인 2D 테이블 형태 `[batch_size, features]`

**주요 특징**:
- `tab-transformer-pytorch` 라이브러리 활용
- 범주형/연속형 변수 최적화된 처리
- 순차적 데이터 분할 (시계열 특성 고려)

## 🔧 환경 설정 및 라이브러리 import

In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import yaml
import logging
import json
from datetime import datetime
from tqdm import tqdm

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# PyTorch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

# TabTransformer
from tab_transformer_pytorch import TabTransformer

## 📊 유틸리티 함수들

### 설정 및 시드 관리

In [ ]:
def load_config(config_path):
    """YAML 설정 파일들을 통합하여 로드"""
    configs = {}

    # 모든 설정 파일 로드
    config_files = ["dataset", "model", "training"]

    for file in config_files:
        with open(f"configs/{file}.yaml", "r") as f:
            config = yaml.safe_load(f)
            configs.update(config)

    return configs


def set_random_seeds(seed=42):
    """재현성을 위한 랜덤 시드 설정"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 🗂️ 데이터셋 클래스 및 전처리

### TabularDataset 클래스

In [ ]:
class TabularDataset(Dataset):
    """TabTransformer를 위한 테이블 형태 데이터셋"""
    
    def __init__(self, categorical_data, continuous_data, targets):
        self.categorical_data = torch.tensor(categorical_data, dtype=torch.long)
        self.continuous_data = torch.tensor(continuous_data, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return {
            "categorical": self.categorical_data[idx],
            "continuous": self.continuous_data[idx],
            "target": self.targets[idx],
        }

### 순차적 데이터 분할 함수

In [ ]:
def sequential_split(data, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    """시계열 특성을 고려한 순차적 데이터 분할"""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "비율의 합이 1.0이어야 합니다"

    n = len(data)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_data = data[:train_end]
    val_data = data[train_end:val_end]
    test_data = data[val_end:]

    return train_data, val_data, test_data


def preprocess(config):
    """Excel 데이터 로드 및 기본 전처리"""
    # Excel 파일의 모든 시트 로드 (header=1)
    data_path = config["file_path"]
    excel = pd.read_excel(data_path, sheet_name=None, header=1)

    # 지정된 시트들 결합
    sheet_names = config["sheet_names"]
    total = pd.concat([excel[sheet_name] for sheet_name in sheet_names])

    # Unnamed: 0 컬럼 제거
    if "Unnamed: 0" in total.columns:
        total.drop(columns="Unnamed: 0", inplace=True)

    # y값 결측치 제거
    original_size = len(total)
    data = total[~total[config["target_column"]].isna()]

    # x값 결측치 제거 (x22-x49 컬럼)
    drop_x_features = config.get("drop_x_features", [f"x{i}" for i in range(22, 49 + 1)])
    if drop_x_features:
        existing_drop_features = [col for col in drop_x_features if col in data.columns]
        if existing_drop_features:
            data = data.drop(columns=existing_drop_features, axis=1)

    # 추가 불필요한 컬럼 제거
    additional_drop_columns = config["additional_drop_columns"]
    if additional_drop_columns:
        existing_additional_drops = [col for col in additional_drop_columns if col in data.columns]
        if existing_additional_drops:
            data = data.drop(columns=existing_additional_drops, axis=1)

    # 인덱스 리셋
    data.reset_index(drop=True, inplace=True)

    return data

### 메인 데이터 로딩 및 전처리 함수

In [ ]:
def load_and_preprocess_data(config):
    """TabTransformer를 위한 데이터 로드 및 전처리"""
    # 데이터 로드
    df = preprocess(config)

    categorical_cols = config["categorical_columns"]
    continuous_cols = config["continuous_columns"]
    target_col = config["target_column"]
    categories = config["categories"]

    print(f"Categorical columns: {categorical_cols}")
    print(f"Continuous columns: {continuous_cols}")
    print(f"Target column: {target_col}")
    print(f"Categories unique values: {categories}")

    # 범주형 데이터 처리
    categorical_data = df[categorical_cols].copy()
    label_encoders = {}

    for i, col in enumerate(categorical_cols):
        le = LabelEncoder()
        categorical_data[col] = le.fit_transform(categorical_data[col])
        label_encoders[col] = le
        print(f"  {col}: {len(le.classes_)} unique values (expected: {categories[i]})")

    # 연속형 데이터 처리
    continuous_data = df[continuous_cols].copy()
    continuous_data = continuous_data.replace(np.inf, 1e5)
    scaler = None
    continuous_mean_std = None

    if config["normalize_continuous"] and len(continuous_cols) > 0:
        scaler = StandardScaler()
        continuous_data = scaler.fit_transform(continuous_data)

        # TabTransformer를 위한 평균/표준편차 계산
        continuous_mean_std = torch.tensor(
            [[scaler.mean_[i], scaler.scale_[i]] for i in range(len(continuous_cols))],
            dtype=torch.float32,
        )

    # 타겟 데이터
    targets = df[target_col].values

    # 순차적 분할 (8:1:1)
    train_ratio = config["train_ratio"]
    val_ratio = config["val_ratio"]
    test_ratio = config["test_ratio"]

    # 분할을 위한 데이터 준비
    X_cat = categorical_data.values
    X_cont = continuous_data if isinstance(continuous_data, np.ndarray) else continuous_data.values

    # 순차적 분할
    X_cat_train, X_cat_val, X_cat_test = sequential_split(X_cat, train_ratio, val_ratio, test_ratio)
    X_cont_train, X_cont_val, X_cont_test = sequential_split(X_cont, train_ratio, val_ratio, test_ratio)
    y_train, y_val, y_test = sequential_split(targets, train_ratio, val_ratio, test_ratio)

    # 데이터셋 생성
    train_dataset = TabularDataset(X_cat_train, X_cont_train, y_train)
    val_dataset = TabularDataset(X_cat_val, X_cont_val, y_val)
    test_dataset = TabularDataset(X_cat_test, X_cont_test, y_test)

    return {
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "test_dataset": test_dataset,
        "categories": categories,
        "num_continuous": len(continuous_cols),
        "continuous_mean_std": continuous_mean_std,
        "label_encoders": label_encoders,
        "scaler": scaler,
    }

### 커스텀 배치 샘플러

In [ ]:
class SequentialBatchSampler:
    """배치 내 순서는 유지하되 배치 간에는 셔플하는 샘플러"""

    def __init__(self, dataset_size, batch_size, shuffle_batches=True):
        self.dataset_size = dataset_size
        self.batch_size = batch_size
        self.shuffle_batches = shuffle_batches

    def __iter__(self):
        # 순차적 배치 생성
        batches = []
        for i in range(0, self.dataset_size, self.batch_size):
            batch = list(range(i, min(i + self.batch_size, self.dataset_size)))
            batches.append(batch)

        # 배치 순서 셔플 (요청시)
        if self.shuffle_batches:
            np.random.shuffle(batches)

        # 인덱스 생성
        for batch in batches:
            for idx in batch:
                yield idx

    def __len__(self):
        return self.dataset_size


def create_dataloaders(datasets, batch_size):
    """TabTransformer용 PyTorch DataLoader 생성"""

    # 훈련용: 배치 셔플, 배치 내 순서 유지
    train_sampler = SequentialBatchSampler(
        len(datasets["train_dataset"]), batch_size, shuffle_batches=True
    )

    train_loader = DataLoader(
        datasets["train_dataset"],
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=4,
        drop_last=True,  # 마지막 불완전한 배치 제거
    )

    # 검증 및 테스트용: 셔플 없음
    val_loader = DataLoader(
        datasets["val_dataset"], batch_size=batch_size, shuffle=False, num_workers=4
    )

    test_loader = DataLoader(
        datasets["test_dataset"], batch_size=batch_size, shuffle=False, num_workers=4
    )

    return train_loader, val_loader, test_loader

## 🏗️ TabTransformer 모델 구조

### 모델 생성 함수

In [ ]:
def create_model(config, categories, num_continuous, continuous_mean_std=None):
    """TabTransformer 모델 생성"""

    # 활성화 함수 설정
    if config["mlp_act"] == "ReLU":
        mlp_act = nn.ReLU()
    elif config["mlp_act"] == "GELU":
        mlp_act = nn.GELU()
    elif config["mlp_act"] == "SELU":
        mlp_act = nn.SELU()
    else:
        mlp_act = nn.ReLU()

    model = TabTransformer(
        categories=tuple(categories),
        num_continuous=num_continuous,
        dim=config["dim"],
        dim_out=config["dim_out"],
        depth=config["depth"],
        heads=config["heads"],
        dim_head=config["dim_head"],
        attn_dropout=config["attn_dropout"],
        ff_dropout=config["ff_dropout"],
        mlp_hidden_mults=tuple(config["mlp_hidden_mults"]),
        mlp_act=mlp_act,
        continuous_mean_std=continuous_mean_std,
    )

    return model

## 🚂 훈련 관련 함수들

### 옵티마이저 및 스케줄러 설정

In [ ]:
def setup_optimizer_and_scheduler(model, config):
    """옵티마이저와 스케줄러 설정"""
    optimizer = AdamW(
        model.parameters(),
        lr=config["training"]["learning_rate"],
        weight_decay=config["training"]["weight_decay"],
    )

    if config["training"]["scheduler"] == "ReduceLROnPlateau":
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=config["training"]["scheduler_patience"],
            factor=0.5,
            verbose=True,
        )
    elif config["training"]["scheduler"] == "StepLR":
        scheduler = StepLR(optimizer, step_size=30, gamma=0.1)
    else:
        scheduler = None

    return optimizer, scheduler

### 훈련 에폭 함수

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device):
    """한 에폭 훈련"""
    model.train()
    total_loss = 0
    num_batches = len(dataloader)

    for batch in dataloader:
        categorical = batch["categorical"].to(device)
        continuous = batch["continuous"].to(device)
        targets = batch["target"].to(device).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(categorical, continuous)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / num_batches


def train_epoch_with_progress(model, dataloader, optimizer, criterion, device, epoch):
    """진행바가 있는 훈련 에폭"""
    model.train()
    total_loss = 0

    # 배치별 진행바
    batch_pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False, unit="batch")

    for batch_idx, batch in enumerate(batch_pbar):
        continuous_data = batch["continuous"].to(device)
        categorical_data = batch["categorical"].to(device)
        targets = batch["target"].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(categorical_data, continuous_data)
        loss = criterion(outputs, targets)

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 배치 진행바 업데이트
        batch_pbar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "Avg Loss": f"{total_loss / (batch_idx + 1):.4f}",
        })

    batch_pbar.close()
    return total_loss / len(dataloader)

### 검증 에폭 함수

In [ ]:
def validate_epoch(model, dataloader, criterion, device, metrics_to_calculate=["mae", "mape"]):
    """검증 에폭"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch in dataloader:
            categorical = batch["categorical"].to(device)
            continuous = batch["continuous"].to(device)
            targets = batch["target"].to(device).unsqueeze(1)

            outputs = model(categorical, continuous)
            loss = criterion(outputs, targets)

            total_loss += loss.item()

            all_predictions.extend(outputs.cpu().numpy().flatten())
            all_targets.extend(targets.cpu().numpy().flatten())

    # 메트릭 계산
    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)

    metrics = {"loss": total_loss / len(dataloader)}

    if "mae" in metrics_to_calculate:
        metrics["mae"] = mean_absolute_error(all_targets, all_predictions)

    if "mape" in metrics_to_calculate:
        # 0으로 나누기 방지
        mask = all_targets != 0
        if mask.sum() > 0:
            metrics["mape"] = (
                np.mean(np.abs((all_targets[mask] - all_predictions[mask]) / all_targets[mask])) * 100
            )
        else:
            metrics["mape"] = float("inf")

    if "rmse" in metrics_to_calculate:
        mse = mean_squared_error(all_targets, all_predictions)
        metrics["rmse"] = np.sqrt(mse)

    if "mse" in metrics_to_calculate:
        metrics["mse"] = mean_squared_error(all_targets, all_predictions)

    return metrics

### 메인 훈련 루프

In [ ]:
def train_model(model, train_loader, val_loader, config, device):
    """진행바가 있는 메인 훈련 루프"""

    # 로깅 설정
    logging.basicConfig(
        filename="training.log",
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s",
    )

    # 훈련 구성 요소 설정
    criterion = nn.MSELoss()
    optimizer, scheduler = setup_optimizer_and_scheduler(model, config)

    # 훈련 변수
    best_val_metric = float("inf")
    patience_counter = 0
    early_stopping_patience = config["training"]["early_stopping_patience"]

    model.to(device)

    # 전체 에폭 진행바
    epoch_pbar = tqdm(
        range(config["training"]["num_epochs"]),
        desc="Training Progress",
        unit="epoch",
        leave=True,
    )

    for epoch in epoch_pbar:
        # 훈련
        train_loss = train_epoch_with_progress(
            model, train_loader, optimizer, criterion, device, epoch
        )

        # 검증
        val_metrics = validate_epoch(
            model,
            val_loader,
            criterion,
            device,
            metrics_to_calculate=[config["validation"]["metric"], "mape"],
        )

        # 조기 종료 기준 메트릭
        val_metric = val_metrics[config["validation"]["metric"]]

        # 에폭 진행바 업데이트
        epoch_pbar.set_postfix({
            "Train Loss": f"{train_loss:.4f}",
            "Val MAE": f'{val_metrics["mae"]:.4f}',
            "Val MAPE": f'{val_metrics.get("mape", 0):.2f}%',
            "Best": f"{best_val_metric:.4f}",
            "Patience": f"{patience_counter}/{early_stopping_patience}",
        })

        # 스케줄러 업데이트
        if scheduler and config["training"]["scheduler"] == "ReduceLROnPlateau":
            scheduler.step(val_metric)
        elif scheduler:
            scheduler.step()

        # 상세 로깅 (주기적)
        if (epoch + 1) % config["logging"]["log_interval"] == 0:
            log_msg = f"Epoch {epoch+1}/{config['training']['num_epochs']}: "
            log_msg += f"Train Loss: {train_loss:.4f}, "
            log_msg += f"Val MAE: {val_metrics['mae']:.4f}, "
            if "mape" in val_metrics:
                log_msg += f"Val MAPE: {val_metrics['mape']:.4f}"

            tqdm.write(log_msg)
            logging.info(log_msg)

        # 조기 종료 및 모델 저장
        if val_metric < best_val_metric:
            best_val_metric = val_metric
            patience_counter = 0

            # 최고 모델 저장
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_metric": best_val_metric,
                "config": config,
            }, os.path.join(config["exp_dir"], "best_model.pth"))

            tqdm.write(f"New best model saved! Val {config['validation']['metric']}: {best_val_metric:.4f}")

        else:
            patience_counter += 1

        # 정기 모델 저장
        if (epoch + 1) % config["logging"]["save_model_every"] == 0:
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_metric": val_metric,
                "config": config,
            }, os.path.join(config["exp_dir"], f"model_epoch_{epoch+1}.pth"))
            tqdm.write(f"Model checkpoint saved: model_epoch_{epoch+1}.pth")

        # 조기 종료
        if patience_counter >= early_stopping_patience:
            tqdm.write(f"Early stopping triggered after {epoch+1} epochs")
            logging.info(f"Early stopping triggered after {epoch+1} epochs")
            break

    epoch_pbar.close()
    return model

### 모델 평가 함수

In [ ]:
def evaluate_model(model, test_loader, device, metrics=["mae", "mape"]):
    """종합적인 모델 평가"""
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for batch in test_loader:
            categorical = batch["categorical"].to(device)
            continuous = batch["continuous"].to(device)
            targets = batch["target"].to(device)

            outputs = model(categorical, continuous).squeeze()

            all_predictions.extend(outputs.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    # 종합 메트릭 계산
    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)

    result_metrics = {}

    if "mae" in metrics:
        result_metrics["MAE"] = mean_absolute_error(all_targets, all_predictions)

    if "mape" in metrics:
        # 0으로 나누기 방지
        mask = all_targets != 0
        if mask.sum() > 0:
            result_metrics["MAPE"] = (
                np.mean(np.abs((all_targets[mask] - all_predictions[mask]) / all_targets[mask])) * 100
            )
        else:
            result_metrics["MAPE"] = float("inf")

    if "mse" in metrics:
        result_metrics["MSE"] = mean_squared_error(all_targets, all_predictions)

    if "rmse" in metrics:
        mse = mean_squared_error(all_targets, all_predictions)
        result_metrics["RMSE"] = np.sqrt(mse)

    return result_metrics, all_predictions, all_targets


## ⚙️ 설정 파일 구조

### 1. configs/dataset.yaml
```yaml
file_path: "/path/to/your/excel/file.xlsx"
indicies_columns: ['timekey_hr','oper_id']
categorical_columns: ["oper_group", "days", "shift", "x1"]
continuous_columns: ["x2", "x3", ..., "x21"]
target_column: "y"
sheet_names: ["Data_Set1(사외)", "Data_Set2(사외)"]
additional_drop_columns: ["lot_cd", "oper_area"]
train_ratio: 0.8
val_ratio: 0.1
test_ratio: 0.1
normalize_continuous: false
categories: [277, 7, 3, 20]  # 각 범주형 변수의 고유값 개수
```

### 2. configs/model.yaml
```yaml
dim: 32                    # 임베딩 차원
dim_out: 1                 # 출력 차원 (회귀이므로 1)
depth: 6                   # Transformer 레이어 수
heads: 8                   # 어텐션 헤드 수
dim_head: 16               # 각 헤드의 차원
attn_dropout: 0.1          # 어텐션 드롭아웃
ff_dropout: 0.1            # 피드포워드 드롭아웃
mlp_hidden_mults: [4, 2]   # MLP 은닉층 배수
mlp_act: "ReLU"            # MLP 활성화 함수
```

### 3. configs/training.yaml
```yaml
training:
  batch_size: 1024
  learning_rate: 0.001
  num_epochs: 1
  weight_decay: 0.01
  scheduler: "ReduceLROnPlateau"
  scheduler_patience: 10
  early_stopping_patience: 20
  
validation:
  metric: "mae"  # 조기 종료 기준 메트릭
  
evaluation:
  metrics: ["mae", "mape"]  # 평가 메트릭
  
logging:
  log_interval: 10
  save_model_every: 20

## 🎯 메인 실행 함수

In [ ]:
def run():
    """TabTransformer 메인 실행 함수"""
    
    # 명령행 인자 파싱
    import argparse
    
    parser = argparse.ArgumentParser(description="TabTransformer Training Pipeline")
    parser.add_argument("--config-dir", default="configs", help="설정 파일 디렉토리")
    parser.add_argument("--mode", choices=["train", "eval"], default="train", help="실행 모드")
    parser.add_argument("--model-path", default=None, help="평가용 모델 경로")
    parser.add_argument("--output-dir", default="outputs", help="출력 디렉토리")
    parser.add_argument("--exp-name", default=None, help="실험명")
    parser.add_argument("--gpu", type=int, default=None, help="GPU 번호")

    args = parser.parse_args()

    # 랜덤 시드 설정
    set_random_seeds(42)

    # 설정 로드
    config = load_config(args.config_dir)

    # 실험 디렉토리 생성
    if args.exp_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        exp_name = f"exp_{timestamp}"
    else:
        exp_name = args.exp_name

    exp_dir = os.path.join(args.output_dir, exp_name)
    os.makedirs(exp_dir, exist_ok=True)

    print(f"Experiment directory: {exp_dir}")

    # 디바이스 설정
    device = torch.device(f"cuda:{args.gpu}" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 데이터 로드 및 전처리
    print("Loading and preprocessing data...")
    data_info = load_and_preprocess_data(config)

    # 데이터 로더 생성
    train_loader, val_loader, test_loader = create_dataloaders(
        data_info, config["training"]["batch_size"]
    )

    print(f"Data loaded successfully:")
    print(f"  - Categories: {data_info['categories']}")
    print(f"  - Number of continuous features: {data_info['num_continuous']}")
    print(f"  - Train samples: {len(data_info['train_dataset'])}")
    print(f"  - Validation samples: {len(data_info['val_dataset'])}")
    print(f"  - Test samples: {len(data_info['test_dataset'])}")

    # 모델 생성
    model = create_model(
        config,
        data_info["categories"],
        data_info["num_continuous"],
        data_info["continuous_mean_std"],
    )

    print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

    if args.mode == "train":
        # 훈련 모드
        print("Starting training...")

        # 실험 디렉토리를 config에 추가
        config["exp_dir"] = exp_dir

        trained_model = train_model(model, train_loader, val_loader, config, device)

        # 최고 모델 로드
        best_model_path = os.path.join(exp_dir, "best_model.pth")
        checkpoint = torch.load(best_model_path)
        model.load_state_dict(checkpoint["model_state_dict"])

        print("Training completed. Evaluating on test set...")

    else:
        # 평가 모드
        if args.model_path is None:
            raise ValueError("--model-path must be provided in evaluation mode")

        print(f"Loading model from {args.model_path}")
        checkpoint = torch.load(args.model_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])

    # 테스트 셋 최종 평가
    model.to(device)
    eval_metrics = config["evaluation"]["metrics"] if "evaluation" in config else ["mae", "mape"]
    test_metrics, predictions, targets = evaluate_model(model, test_loader, device, eval_metrics)

    print("\nTest Set Results:")
    print("=" * 50)
    for metric, value in test_metrics.items():
        print(f"{metric}: {value:.4f}")

    # 예측 결과 저장
    results_df = pd.DataFrame({
        "actual": targets, 
        "predicted": predictions, 
        "residual": targets - predictions
    })
    predictions_path = os.path.join(exp_dir, "test_predictions.csv")
    results_df.to_csv(predictions_path, index=False)
    print(f"\nPredictions saved to '{predictions_path}'")

    # 테스트 결과 JSON 저장
    test_results = {
        "experiment_info": {
            "exp_name": exp_name,
            "exp_dir": exp_dir,
            "timestamp": datetime.now().isoformat(),
            "mode": args.mode,
            "model_path": (
                args.model_path if args.mode == "eval" 
                else os.path.join(exp_dir, "best_model.pth")
            ),
        },
        "dataset_info": {
            "file_path": config["file_path"],
            "categories": data_info["categories"],
            "num_continuous": data_info["num_continuous"],
            "train_samples": len(data_info["train_dataset"]),
            "val_samples": len(data_info["val_dataset"]),
            "test_samples": len(data_info["test_dataset"]),
        },
        "model_info": {
            "total_parameters": sum(p.numel() for p in model.parameters()),
            "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        },
        "test_metrics": test_metrics,
        "config": config,
    }

    results_path = os.path.join(exp_dir, "test_results.json")
    with open(results_path, "w") as f:
        json.dump(test_results, f, indent=2, default=str)
    print(f"Test results saved to '{results_path}'")

    # 설정 백업
    config_path = os.path.join(exp_dir, "config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2, default=str)
    print(f"Configuration saved to '{config_path}'")

    print(f"\nAll outputs saved in: {exp_dir}")


# 노트북에서 직접 실행
if __name__ == "__main__":
    run()

## 💡 하이퍼파라미터 튜닝 가이드

### 1. 모델 크기 조정 (model.yaml)

**기본 설정 (중간 성능)**:
```yaml
dim: 32
depth: 6
heads: 8
```

**작은 모델 (메모리 절약)**:
```yaml
dim: 16
depth: 4
heads: 4
mlp_hidden_mults: [2, 1]
```

**큰 모델 (고성능)**:
```yaml
dim: 64
depth: 8
heads: 16
mlp_hidden_mults: [6, 4]
```

### 2. 학습률 스케줄링

**안정적인 학습**:
```yaml
learning_rate: 0.001
scheduler: "ReduceLROnPlateau"
scheduler_patience: 10
```

**빠른 학습**:
```yaml
learning_rate: 0.003
scheduler: "StepLR"
```

### 4. 정규화 조정

**과적합 방지**:
```yaml
attn_dropout: 0.2
ff_dropout: 0.2
weight_decay: 0.1
```

## 📝 실행 예시

### 기본 실행
```bash
python main.py
```

### 사용자 정의 설정
```bash
python main.py --gpu 1 --exp-name tat_prediction_v1
```

### 평가 모드
```bash
python  main.py --mode eval --model-path ./outputs/exp_20241201_120000/best_model.pth

## ❗ 문제 해결

### 자주 발생하는 문제들

#### 1. 메모리 부족 오류
```
RuntimeError: CUDA out of memory
```
**해결방법:**
- `training.yaml`에서 `batch_size`를 줄여보세요 (예: 1024 → 512)
- `model.yaml`에서 `dim`을 줄여보세요 (예: 32 → 16)

#### 2. 범주형 변수 개수 불일치
```
ValueError: categories mismatch
```
**해결방법:**
```python
# 실제 데이터의 범주형 변수 고유값 개수 확인
import pandas as pd
df = pd.read_excel('your_file.xlsx', sheet_name='Data_Set1(사외)', header=1)
for col in ['oper_group', 'days', 'shift', 'x1']:
    print(f"{col}: {df[col].nunique()} unique values")
```